# 🎨 Module 12.2: Autonomous Photo Editor with Reinforcement Learning

### *Teaching an AI Agent to Transform Amateur Photos into Professional-Quality Images*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_12_Real_World_Projects/02_Autonomous_Image_Editor/autonomous_image_editor.ipynb)

---

**Course**: Reinforcement Learning for Image Processing 
**Module**: 12 – Real-World Projects 
**Project**: 02 – Autonomous Image Editor 

---

## 🎯 Learning Objectives

By completing this project, you will:

1. **Formulate** photo editing as a sequential decision-making (MDP) problem
2. **Implement** computational aesthetic quality metrics (colorfulness, contrast, sharpness, exposure)
3. **Design** a discrete action space covering common photo editing operations
4. **Build** a DQN agent that learns to improve photo quality autonomously
5. **Evaluate** editing sequences and analyze learned editing strategies
6. **Visualize** before/after comparisons and training dynamics

## 📦 Environment Setup

## Deep Derivation: Autonomous Image Editing via Sequential Decision Making

### Step 1: Image Editing as an MDP with Composable Actions
Define the editing MDP $(\mathcal{S}, \mathcal{A}, T, R, \gamma)$:

**State:** $s_t = (I_t, I_{\text{target}}, h_t, t/T)$ where $I_t$ is the current image, $I_{\text{target}}$ is the goal (or text description), $h_t$ is the edit history, and $t/T$ is the normalized time step.

**Action space (composable edits):**
$$\mathcal{A} = \{(\text{op}_k, \theta_k) : k = 1, \ldots, K\}$$

where $\text{op}_k \in \{$brightness, contrast, saturation, hue, sharpen, blur, crop, rotate$\}$ and $\theta_k \in \mathbb{R}^{d_k}$ are continuous parameters.

**Key property — composition:** $I_T = a_T \circ a_{T-1} \circ \cdots \circ a_1(I_0)$

Most editing operations are NOT commutative: $\text{brightness} \circ \text{contrast} \neq \text{contrast} \circ \text{brightness}$. This means the ORDER matters — a key reason RL is superior to one-shot prediction.

### Step 2: Differentiable Image Signal Processing (ISP) Pipeline
Each editing operation is a differentiable function:

**Brightness:** $I' = I + \beta, \quad \nabla_\beta I' = \mathbf{1}$

**Contrast:** $I' = \alpha(I - \mu) + \mu, \quad \nabla_\alpha I' = I - \mu$

**Gamma correction:** $I' = I^\gamma, \quad \nabla_\gamma I' = I^\gamma \ln I$

**Saturation (in HSV):** Convert $I_{\text{RGB}} \to (H, S, V)$:
$$S' = \text{clip}(s \cdot S, 0, 1)$$

**Derivation of end-to-end gradient:** By the chain rule through $N$ sequential operations:
$$\frac{\partial I_N}{\partial \theta_1} = \frac{\partial f_N}{\partial I_{N-1}} \cdot \frac{\partial f_{N-1}}{\partial I_{N-2}} \cdots \frac{\partial f_2}{\partial I_1} \cdot \frac{\partial f_1}{\partial \theta_1} = \left(\prod_{k=2}^N J_k\right) \cdot \frac{\partial f_1}{\partial \theta_1}$$

where $J_k = \frac{\partial f_k}{\partial I_{k-1}}$ is the Jacobian of each operation. This product can cause vanishing/exploding gradients for long editing sequences — motivating RL over direct backpropagation. $\blacksquare$

### Step 3: Color Space Transformations (Mathematical Detail)
**RGB to CIE Lab (perceptually uniform):**

Step 1: RGB → XYZ (linear transformation):
$$\begin{pmatrix} X \\ Y \\ Z \end{pmatrix} = M_{sRGB} \begin{pmatrix} R_{\text{linear}} \\ G_{\text{linear}} \\ B_{\text{linear}} \end{pmatrix}$$

where $R_{\text{linear}} = \begin{cases} R/12.92 & R \leq 0.04045 \\ ((R+0.055)/1.055)^{2.4} & \text{otherwise} \end{cases}$

Step 2: XYZ → Lab:
$$L^* = 116 f(Y/Y_n) - 16, \quad a^* = 500(f(X/X_n) - f(Y/Y_n)), \quad b^* = 200(f(Y/Y_n) - f(Z/Z_n))$$

where $f(t) = \begin{cases} t^{1/3} & t > \delta^3 \\ t/(3\delta^2) + 4/29 & \text{otherwise} \end{cases}$, $\delta = 6/29$.

**Why Lab for editing?** The perceptual distance $\Delta E = \sqrt{(\Delta L^*)^2 + (\Delta a^*)^2 + (\Delta b^*)^2}$ approximates human color difference perception:
- $\Delta E < 1$: imperceptible
- $\Delta E \approx 2.3$: just noticeable difference (JND)
- $\Delta E > 5$: clearly different

This makes Lab-space rewards more perceptually meaningful than RGB-space rewards. $\blacksquare$

### Step 4: Reward Function for Autonomous Editing
**Multi-component reward:**
$$R(I_t, a_t, I_{\text{target}}) = w_1 R_{\text{pixel}} + w_2 R_{\text{perceptual}} + w_3 R_{\text{aesthetic}} + w_4 R_{\text{efficiency}}$$

**Pixel reward:** $R_{\text{pixel}} = -\frac{\|I_t - I_{\text{target}}\|_1}{H \times W \times C}$ (normalized L1)

**Perceptual reward:** $R_{\text{perceptual}} = -\text{LPIPS}(I_t, I_{\text{target}})$

**Aesthetic reward:** $R_{\text{aesthetic}} = \text{NIMA}(I_t) / 10$ (normalized to [0,1])

**Efficiency penalty:** $R_{\text{efficiency}} = -\lambda \cdot t / T_{\text{max}}$

**Derivation of weight balancing:** Use automatic multi-objective scaling:
$$w_i = \frac{1}{\sigma_i^2}$$

where $\sigma_i^2 = \text{Var}(R_i)$ estimated from a rolling window. This ensures each reward component contributes equally to the gradient magnitude, regardless of scale.

**Proof:** The gradient of the weighted sum $\sum w_i R_i$ has equal contribution from each component when $w_i \propto 1/\text{Var}(R_i)$, since $\text{Var}(w_i R_i) = w_i^2 \text{Var}(R_i) = 1/\text{Var}(R_i) \cdot \text{Var}(R_i) = 1$. $\blacksquare$

### Step 5: SAC (Soft Actor-Critic) for Continuous Editing Actions
SAC maximizes entropy-augmented return:
$$J(\pi) = \sum_t E\left[\gamma^t\left(r_t + \alpha H(\pi(\cdot|s_t))\right)\right]$$

**Soft Q-function:**
$$Q^{\text{soft}}(s,a) = r(s,a) + \gamma E_{s'}\left[V^{\text{soft}}(s')\right]$$
$$V^{\text{soft}}(s) = E_{a\sim\pi}[Q^{\text{soft}}(s,a) - \alpha\log\pi(a|s)]$$

**Policy update (minimizing KL divergence):**
$$\pi^* = \arg\min_\pi D_{KL}\left(\pi(\cdot|s) \middle\| \frac{\exp(Q^{\text{soft}}(s,\cdot)/\alpha)}{Z(s)}\right)$$

**Derivation of automatic temperature tuning:**
$$\alpha^* = \arg\min_\alpha E_{a\sim\pi^*}\left[-\alpha\log\pi^*(a|s) - \alpha\bar{H}\right]$$

where $\bar{H}$ is the target entropy (typically $-\dim(\mathcal{A})$).

Taking derivative: $\frac{\partial}{\partial\alpha} = E[-\log\pi^*(a|s)] - \bar{H}$

Setting to zero: $E[-\log\pi^*(a|s)] = \bar{H}$

This automatically adjusts exploration: if $H(\pi) > \bar{H}$, decrease $\alpha$ (less exploration); if $H(\pi) < \bar{H}$, increase $\alpha$ (more exploration). $\blacksquare$

### Step 6: Inverse RL for Learning from Expert Edits
Given expert editing trajectories $\mathcal{D} = \{\tau_1, \ldots, \tau_N\}$, recover the reward function:
$$R^* = \arg\max_R \left[E_{\tau \sim \mathcal{D}}[\sum_t R(s_t, a_t)] - \log E_{\tau \sim \pi_R}[\exp(\sum_t R(s_t, a_t))]\right]$$

**Maximum Entropy IRL (Ziebart et al., 2008):**
$$p(\tau | R) = \frac{1}{Z}\exp\left(\sum_t R(s_t, a_t)\right)$$

**Derivation:** The likelihood of expert demonstrations under the MaxEnt model:
$$\log p(\mathcal{D} | R) = \sum_{\tau \in \mathcal{D}} \sum_t R(s_t, a_t) - |\mathcal{D}| \log Z$$

The gradient:
$$\nabla_R \log p = \underbrace{E_{\mathcal{D}}\left[\sum_t \nabla_R R(s_t,a_t)\right]}_{\text{expert feature expectations}} - \underbrace{E_{\pi_R}\left[\sum_t \nabla_R R(s_t,a_t)\right]}_{\text{policy feature expectations}}$$

At convergence, the policy's feature expectations match the expert's. $\blacksquare$

### Step 7: Undo/Redo as Temporal Abstraction
Allow the agent to undo previous actions:
$$\mathcal{A}_{\text{ext}} = \mathcal{A} \cup \{\text{undo}_k : k = 1, \ldots, K\}$$

where $\text{undo}_k$ reverts the last $k$ actions.

**Derivation of value of undo:** The value of having undo available:
$$V_{\text{with\_undo}}(s) - V_{\text{without\_undo}}(s) = E\left[\max(0, V(s_{t-k}) - V(s_t) - k \cdot c_{\text{undo}})\right]$$

where $c_{\text{undo}}$ is a small cost per undo step.

**Optimal undo depth:** The agent should undo to step $k^*$:
$$k^* = \arg\max_k \left[V(s_{t-k}) - k \cdot c_{\text{undo}}\right]$$

This is a discrete optimization over the edit history. With $c_{\text{undo}} > 0$, the agent prefers minimal undos — encouraging it to make good decisions initially rather than relying on trial-and-error with undo. $\blacksquare$

In [ ]:
!pip install -q torch torchvision numpy matplotlib scipy Pillow

In [ ]:
# Download REAL open-source datasets for real-world projects
import torchvision
import subprocess
import sys

# MedMNIST for medical imaging (install + download)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'medmnist', '-q'])
from medmnist import ChestMNIST, DermaMNIST

# Chest X-rays (real medical images!)
chest_data = ChestMNIST(split='train', download=True, size=28)
print(f"✅ ChestMNIST: {len(chest_data)} real chest X-ray images")

# Dermatology images
derma_data = DermaMNIST(split='train', download=True, size=28)
print(f"✅ DermaMNIST: {len(derma_data)} real skin lesion images")

# EuroSAT for satellite imagery
try:
    eurosat = torchvision.datasets.EuroSAT(root='./data', download=True)
    print(f"✅ EuroSAT: {len(eurosat)} real satellite images (64x64, 10 land-use classes)")
except:
    print("⚠️ EuroSAT downloading...")

# CIFAR-10 for video/editing projects
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos for editing/video projects")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque, namedtuple
from scipy import ndimage
from PIL import Image
import random
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("Setup complete!")

---

## 1. 📚 Introduction: Autonomous Photo Editing

### From Amateur to Professional

Professional photographers spend significant time in post-processing to transform raw captures into stunning images. This involves adjusting **brightness**, **contrast**, **saturation**, **sharpness**, applying **color grading**, and **cropping** for better composition. Can we train an RL agent to learn these skills?

### Sequential Editing as an RL Problem

Photo editing is inherently sequential — each adjustment builds upon previous ones, and the order matters. This maps naturally to a **Markov Decision Process (MDP)**:

- The agent observes the current image state
- Selects an editing action from a discrete set
- Receives a reward based on aesthetic quality improvement
- The environment transitions to the modified image

### Aesthetic Quality as Reward

We use a computational aesthetic quality function that combines multiple perceptual metrics into a single scalar reward, approximating human aesthetic preferences.

---

## 2. 📝 Problem Formulation

### MDP Definition

We define the photo editing MDP as:

| Component | Definition |
|-----------|------------|
| **State** $s_t$ | Current image pixels + edit history encoding |
| **Action** $a_t$ | Discrete editing operation |
| **Transition** $T$ | Deterministic image transformation |
| **Reward** $R_t$ | Change in aesthetic quality score |
| **Terminal** | `done` action or max steps reached |

### Action Space

The agent can select from 13 discrete actions:

$$\mathcal{A} = \{\text{crop\_center}, \text{brightness}_{\pm}, \text{contrast}_{\pm}, \text{saturation}_{\pm}, \text{sharpen}, \text{blur}, \text{warm}, \text{cool}, \text{vignette}, \text{done}\}$$

Each action modifies the image by a **fixed increment**, making the environment deterministic.

### Aesthetic Quality Score

Our composite aesthetic metric is:

$$Q_{\text{aesthetic}} = w_1 \cdot \text{colorfulness} + w_2 \cdot \text{contrast\_score} + w_3 \cdot \text{sharpness} + w_4 \cdot \text{exposure\_balance} + w_5 \cdot \text{rule\_of\_thirds}$$

### Colorfulness Metric (Hasler & Süsstrunk, 2003)

$$C = \sqrt{\sigma_{rg}^2 + \sigma_{yb}^2} + 0.3\sqrt{\mu_{rg}^2 + \mu_{yb}^2}$$

where $rg = R - G$ and $yb = 0.5(R+G) - B$ are opponent color channels.

### Reward Function

$$R_t = Q_{\text{aesthetic}}(I_t) - Q_{\text{aesthetic}}(I_{t-1}) - \lambda \cdot t$$

The time penalty $\lambda \cdot t$ encourages the agent to achieve improvements efficiently, discouraging unnecessary edits.

### Contrast Score (RMS Contrast)

$$C_{\text{RMS}} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(L_i - \bar{L})^2}$$

where $L_i$ is the luminance of pixel $i$ and $\bar{L}$ is the mean luminance.

### Sharpness (Laplacian Variance)

$$S = \text{Var}(\nabla^2 I) = \text{Var}\left(\frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}\right)$$

---

## 3. 📊 Aesthetic Quality Metrics Implementation

In [ ]:
class AestheticScorer:
    """Computational aesthetic quality assessment for photographs."""

    def __init__(self, weights=None):
        if weights is None:
            self.weights = {
                'colorfulness': 0.25,
                'contrast': 0.20,
                'sharpness': 0.20,
                'exposure': 0.20,
                'rule_of_thirds': 0.15
            }
        else:
            self.weights = weights

    def colorfulness(self, image):
        """Hasler & Susstrunk colorfulness metric."""
        R, G, B = image[:,:,0], image[:,:,1], image[:,:,2]
        rg = R.astype(float) - G.astype(float)
        yb = 0.5 * (R.astype(float) + G.astype(float)) - B.astype(float)

        sigma_rg = np.std(rg)
        sigma_yb = np.std(yb)
        mu_rg = np.mean(rg)
        mu_yb = np.mean(yb)

        C = np.sqrt(sigma_rg**2 + sigma_yb**2) + 0.3 * np.sqrt(mu_rg**2 + mu_yb**2)
        return np.clip(C / 100.0, 0, 1)  # Normalize to [0, 1]

    def contrast_score(self, image):
        """RMS contrast on luminance channel."""
        luminance = 0.2126 * image[:,:,0] + 0.7152 * image[:,:,1] + 0.0722 * image[:,:,2]
        rms = np.std(luminance.astype(float))
        return np.clip(rms / 80.0, 0, 1)  # Normalize

    def sharpness(self, image):
        """Laplacian variance as sharpness measure."""
        gray = 0.2126 * image[:,:,0] + 0.7152 * image[:,:,1] + 0.0722 * image[:,:,2]
        laplacian = ndimage.laplace(gray.astype(float))
        score = np.var(laplacian)
        return np.clip(score / 500.0, 0, 1)  # Normalize

    def exposure_balance(self, image):
        """How well-exposed is the image (penalizes over/under exposure)."""
        luminance = 0.2126 * image[:,:,0] + 0.7152 * image[:,:,1] + 0.0722 * image[:,:,2]
        mean_lum = np.mean(luminance) / 255.0
        # Optimal exposure around 0.45-0.55
        score = 1.0 - 4.0 * (mean_lum - 0.5)**2
        return np.clip(score, 0, 1)

    def rule_of_thirds(self, image):
        """Approximate rule-of-thirds score based on edge energy distribution."""
        gray = 0.2126 * image[:,:,0] + 0.7152 * image[:,:,1] + 0.0722 * image[:,:,2]
        edges = ndimage.sobel(gray.astype(float))
        h, w = edges.shape

        # Rule of thirds intersection points
        thirds_h = [h//3, 2*h//3]
        thirds_w = [w//3, 2*w//3]

        margin_h = max(h // 12, 1)
        margin_w = max(w // 12, 1)

        total_energy = np.sum(edges) + 1e-8
        thirds_energy = 0

        for th in thirds_h:
            for tw in thirds_w:
                region = edges[max(0,th-margin_h):th+margin_h, max(0,tw-margin_w):tw+margin_w]
                thirds_energy += np.sum(region)

        score = thirds_energy / total_energy
        return np.clip(score * 3.0, 0, 1)  # Scale up

    def compute_score(self, image):
        """Compute composite aesthetic score."""
        scores = {
            'colorfulness': self.colorfulness(image),
            'contrast': self.contrast_score(image),
            'sharpness': self.sharpness(image),
            'exposure': self.exposure_balance(image),
            'rule_of_thirds': self.rule_of_thirds(image)
        }

        total = sum(self.weights[k] * scores[k] for k in scores)
        return total, scores

scorer = AestheticScorer()
print("Aesthetic Scorer initialized.")
print(f"Weights: {scorer.weights}")

### Visualize Metric Behavior

Let's understand how each metric responds to different image properties.

In [ ]:
def create_test_images():
    """Create test images to demonstrate metric behavior."""
    size = (64, 64, 3)

    # Low contrast, dull image
    low_contrast = np.random.randint(100, 155, size=size).astype(np.uint8)

    # High contrast, colorful image
    high_contrast = np.zeros(size, dtype=np.uint8)
    high_contrast[:32, :32] = [255, 50, 50]
    high_contrast[:32, 32:] = [50, 255, 50]
    high_contrast[32:, :32] = [50, 50, 255]
    high_contrast[32:, 32:] = [255, 255, 50]

    # Overexposed
    overexposed = np.random.randint(200, 255, size=size).astype(np.uint8)

    # Underexposed
    underexposed = np.random.randint(0, 55, size=size).astype(np.uint8)

    # Well-balanced
    balanced = np.random.randint(60, 200, size=size).astype(np.uint8)
    # Add some structure
    balanced[20:44, 20:44] = [200, 100, 50]
    balanced[10:20, 40:54] = [50, 180, 200]

    return {
        'Low Contrast/Dull': low_contrast,
        'High Contrast/Colorful': high_contrast,
        'Overexposed': overexposed,
        'Underexposed': underexposed,
        'Well-Balanced': balanced
    }

test_images = create_test_images()

fig, axes = plt.subplots(2, 5, figsize=(16, 7))

for idx, (name, img) in enumerate(test_images.items()):
    axes[0, idx].imshow(img)
    axes[0, idx].set_title(name, fontsize=10, fontweight='bold')
    axes[0, idx].axis('off')

    total, scores = scorer.compute_score(img)

    metrics = list(scores.keys())
    values = list(scores.values())
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(metrics)))

    bars = axes[1, idx].bar(range(len(metrics)), values, color=colors)
    axes[1, idx].set_xticks(range(len(metrics)))
    axes[1, idx].set_xticklabels(['Color', 'Contr', 'Sharp', 'Expo', 'RoT'], fontsize=8)
    axes[1, idx].set_ylim(0, 1)
    axes[1, idx].set_title(f'Total: {total:.3f}', fontsize=10)
    axes[1, idx].axhline(y=0.5, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Aesthetic Quality Metrics Across Different Image Types', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 4. 📷 Synthetic Photo Generation

We generate synthetic "amateur" photographs with common quality issues:
- Poor exposure (too dark or too bright)
- Low contrast
- Desaturated colors
- Slight blur

In [ ]:
class SyntheticPhotoGenerator:
    """Generate synthetic amateur photographs with controllable quality defects."""

    def __init__(self, size=(64, 64)):
        self.size = size

    def generate_scene(self):
        """Generate a base scene with objects and background."""
        h, w = self.size
        image = np.zeros((h, w, 3), dtype=np.float32)

        # Sky gradient background
        sky_top = np.array([0.3, 0.5, 0.9])
        sky_bottom = np.array([0.6, 0.75, 0.95])
        for i in range(h // 2):
            t = i / (h // 2)
            image[i, :] = sky_top * (1-t) + sky_bottom * t

        # Ground
        ground_color = np.array([0.3, 0.6, 0.2]) + np.random.uniform(-0.1, 0.1, 3)
        ground_color = np.clip(ground_color, 0, 1)
        image[h//2:, :] = ground_color

        # Add gradient to ground
        for i in range(h//2, h):
            t = (i - h//2) / (h//2)
            image[i, :] *= (1.0 - 0.3*t)

        # Add random objects (circles/rectangles)
        n_objects = np.random.randint(2, 5)
        for _ in range(n_objects):
            cx, cy = np.random.randint(10, w-10), np.random.randint(10, h-10)
            radius = np.random.randint(5, 15)
            color = np.random.uniform(0.2, 0.9, 3)

            y, x = np.ogrid[:h, :w]
            mask = ((x - cx)**2 + (y - cy)**2) <= radius**2
            image[mask] = color

        # Add a "subject" slightly off-center (amateur composition)
        offset_x = np.random.randint(-10, 10)
        offset_y = np.random.randint(-5, 5)
        sx, sy = w//2 + offset_x, h//2 + offset_y
        subject_size = np.random.randint(8, 16)
        subject_color = np.random.uniform(0.5, 1.0, 3)

        y, x = np.ogrid[:h, :w]
        mask = ((x - sx)**2 + (y - sy)**2) <= subject_size**2
        image[mask] = subject_color

        return np.clip(image, 0, 1)

    def apply_amateur_defects(self, image):
        """Apply common amateur photography defects."""
        defects = []

        # Random exposure issues
        exposure_shift = np.random.uniform(-0.25, 0.15)
        image = image + exposure_shift
        if abs(exposure_shift) > 0.1:
            defects.append('exposure')

        # Reduce contrast
        if np.random.random() > 0.3:
            mean = image.mean()
            image = mean + (image - mean) * np.random.uniform(0.5, 0.75)
            defects.append('low_contrast')

        # Desaturate
        if np.random.random() > 0.3:
            gray = image.mean(axis=2, keepdims=True)
            sat_factor = np.random.uniform(0.3, 0.6)
            image = gray + (image - gray) * sat_factor
            defects.append('desaturated')

        # Slight blur
        if np.random.random() > 0.4:
            sigma = np.random.uniform(0.5, 1.5)
            for c in range(3):
                image[:,:,c] = ndimage.gaussian_filter(image[:,:,c], sigma)
            defects.append('blur')

        # Add noise
        noise = np.random.normal(0, 0.02, image.shape)
        image = image + noise

        image = np.clip(image, 0, 1)
        return image, defects

    def generate(self):
        """Generate a single amateur photo."""
        scene = self.generate_scene()
        amateur_photo, defects = self.apply_amateur_defects(scene)
        return (amateur_photo * 255).astype(np.uint8), defects

generator = SyntheticPhotoGenerator(size=(64, 64))
print("Photo generator initialized.")

In [ ]:
# Generate and visualize sample amateur photos
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

for i in range(10):
    row, col = i // 5, i % 5
    photo, defects = generator.generate()
    total_score, _ = scorer.compute_score(photo)

    axes[row, col].imshow(photo)
    axes[row, col].set_title(f'Score: {total_score:.3f}\n{", ".join(defects[:2])}', fontsize=9)
    axes[row, col].axis('off')

plt.suptitle('Sample Amateur Photos with Aesthetic Scores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 5. 🎮 Photo Editing Environment

### Environment Design

The editing environment wraps the image and exposes a Gym-like interface:

In [ ]:
class PhotoEditingEnv:
    """
    RL Environment for autonomous photo editing.

    State: downsampled image features + edit history
    Actions: discrete editing operations
    Reward: aesthetic quality improvement - time penalty
    """

    ACTION_NAMES = [
        'crop_center',
        'increase_brightness',
        'decrease_brightness',
        'increase_contrast',
        'decrease_contrast',
        'increase_saturation',
        'decrease_saturation',
        'sharpen',
        'blur_smooth',
        'warm_filter',
        'cool_filter',
        'vignette',
        'no_op_done'
    ]

    def __init__(self, max_steps=15, time_penalty=0.002, image_size=(64, 64)):
        self.max_steps = max_steps
        self.time_penalty = time_penalty
        self.image_size = image_size
        self.n_actions = len(self.ACTION_NAMES)
        self.scorer = AestheticScorer()
        self.generator = SyntheticPhotoGenerator(size=image_size)

        # State dimension: flattened downsampled image (16x16x3) + edit history (13)
        self.state_image_size = (16, 16)
        self.state_dim = 16 * 16 * 3 + self.n_actions

        self.reset()

    def reset(self, image=None):
        """Reset environment with a new or specified image."""
        if image is None:
            self.original_image, _ = self.generator.generate()
        else:
            self.original_image = image.copy()

        self.current_image = self.original_image.copy()
        self.step_count = 0
        self.edit_history = np.zeros(self.n_actions)
        self.prev_score, _ = self.scorer.compute_score(self.current_image)
        self.action_sequence = []

        return self._get_state()

    def _get_state(self):
        """Extract state representation from current image."""
        # Downsample image
        img_pil = Image.fromarray(self.current_image)
        img_small = img_pil.resize(self.state_image_size, Image.BILINEAR)
        img_array = np.array(img_small).flatten().astype(np.float32) / 255.0

        # Concatenate with edit history
        state = np.concatenate([img_array, self.edit_history / (self.max_steps + 1)])
        return state

    def step(self, action):
        """Execute an editing action."""
        self.step_count += 1
        self.edit_history[action] += 1
        self.action_sequence.append(action)

        # Check for done action
        if action == 12:  # no_op/done
            done = True
            reward = 0.0
            return self._get_state(), reward, done, {'action': 'done'}

        # Apply editing action
        self.current_image = self._apply_action(self.current_image, action)

        # Compute reward
        current_score, _ = self.scorer.compute_score(self.current_image)
        reward = current_score - self.prev_score - self.time_penalty * self.step_count
        self.prev_score = current_score

        # Check termination
        done = self.step_count >= self.max_steps

        info = {
            'action': self.ACTION_NAMES[action],
            'score': current_score,
            'step': self.step_count
        }

        return self._get_state(), reward, done, info

    def _apply_action(self, image, action):
        """Apply the specified editing action to the image."""
        img = image.astype(np.float32)

        if action == 0:  # crop_center
            h, w = img.shape[:2]
            margin = max(2, min(h, w) // 16)
            cropped = img[margin:h-margin, margin:w-margin]
            # Resize back
            pil_img = Image.fromarray(cropped.astype(np.uint8))
            pil_img = pil_img.resize((w, h), Image.BILINEAR)
            img = np.array(pil_img).astype(np.float32)

        elif action == 1:  # increase_brightness
            img = img + 15

        elif action == 2:  # decrease_brightness
            img = img - 15

        elif action == 3:  # increase_contrast
            mean = img.mean()
            img = mean + (img - mean) * 1.2

        elif action == 4:  # decrease_contrast
            mean = img.mean()
            img = mean + (img - mean) * 0.85

        elif action == 5:  # increase_saturation
            gray = img.mean(axis=2, keepdims=True)
            img = gray + (img - gray) * 1.3

        elif action == 6:  # decrease_saturation
            gray = img.mean(axis=2, keepdims=True)
            img = gray + (img - gray) * 0.75

        elif action == 7:  # sharpen
            for c in range(3):
                blurred = ndimage.gaussian_filter(img[:,:,c], 1.0)
                img[:,:,c] = img[:,:,c] + 0.5 * (img[:,:,c] - blurred)

        elif action == 8:  # blur_smooth
            for c in range(3):
                img[:,:,c] = ndimage.gaussian_filter(img[:,:,c], 0.7)

        elif action == 9:  # warm_filter
            img[:,:,0] = img[:,:,0] + 10  # More red
            img[:,:,2] = img[:,:,2] - 8   # Less blue

        elif action == 10:  # cool_filter
            img[:,:,0] = img[:,:,0] - 8   # Less red
            img[:,:,2] = img[:,:,2] + 10  # More blue

        elif action == 11:  # vignette
            h, w = img.shape[:2]
            y, x = np.ogrid[:h, :w]
            cx, cy = w/2, h/2
            r = np.sqrt((x - cx)**2 + (y - cy)**2)
            r_max = np.sqrt(cx**2 + cy**2)
            vignette_mask = 1.0 - 0.3 * (r / r_max)**2
            img = img * vignette_mask[:,:,np.newaxis]

        return np.clip(img, 0, 255).astype(np.uint8)

    def get_action_name(self, action):
        return self.ACTION_NAMES[action]

# Test the environment
env = PhotoEditingEnv()
state = env.reset()
print(f"State dimension: {len(state)}")
print(f"Number of actions: {env.n_actions}")
print(f"Action names: {env.ACTION_NAMES}")

In [ ]:
# Demonstrate each action
fig, axes = plt.subplots(2, 7, figsize=(18, 6))

base_photo, _ = generator.generate()

for idx in range(13):
    row, col = idx // 7, idx % 7
    env.reset(image=base_photo)

    if idx == 12:  # done action
        result = base_photo.copy()
    else:
        result = env._apply_action(base_photo.copy(), idx)

    axes[row, col].imshow(result)
    axes[row, col].set_title(env.ACTION_NAMES[idx].replace('_', '\n'), fontsize=8)
    axes[row, col].axis('off')

# Hide the last empty subplot
axes[1, 6].axis('off')

plt.suptitle('Effect of Each Editing Action on a Sample Photo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 6. 🧠 DQN Agent Implementation

We implement a **Deep Q-Network (DQN)** agent with:
- Experience replay buffer
- Target network for stable training
- Epsilon-greedy exploration

In [ ]:
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))


class ReplayBuffer:
    """Fixed-size experience replay buffer."""

    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        transitions = random.sample(self.buffer, batch_size)
        batch = Transition(*zip(*transitions))
        return batch

    def __len__(self):
        return len(self.buffer)


class DQNetwork(nn.Module):
    """Deep Q-Network for photo editing policy."""

    def __init__(self, state_dim, n_actions, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_actions)
        )

    def forward(self, x):
        return self.net(x)


class DQNAgent:
    """DQN Agent for autonomous photo editing."""

    def __init__(self, state_dim, n_actions, lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=500,
                 batch_size=64, target_update=10, buffer_size=10000):

        self.state_dim = state_dim
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update = target_update

        # Epsilon schedule
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.steps_done = 0

        # Networks
        self.policy_net = DQNetwork(state_dim, n_actions).to(device)
        self.target_net = DQNetwork(state_dim, n_actions).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_size)

        # Training metrics
        self.losses = []

    def get_epsilon(self):
        """Compute current epsilon using exponential decay."""
        eps = self.epsilon_end + (self.epsilon_start - self.epsilon_end) * \
              np.exp(-self.steps_done / self.epsilon_decay)
        return eps

    def select_action(self, state, eval_mode=False):
        """Epsilon-greedy action selection."""
        if not eval_mode:
            eps = self.get_epsilon()
            self.steps_done += 1

            if random.random() < eps:
                return random.randrange(self.n_actions)

        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.policy_net(state_tensor)
            return q_values.argmax(dim=1).item()

    def update(self):
        """Perform one step of DQN optimization."""
        if len(self.buffer) < self.batch_size:
            return None

        batch = self.buffer.sample(self.batch_size)

        states = torch.FloatTensor(np.array(batch.state)).to(device)
        actions = torch.LongTensor(batch.action).to(device)
        rewards = torch.FloatTensor(batch.reward).to(device)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(device)
        dones = torch.FloatTensor(batch.done).to(device)

        # Current Q-values
        current_q = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Target Q-values
        with torch.no_grad():
            next_q = self.target_net(next_states).max(1)[0]
            target_q = rewards + self.gamma * next_q * (1 - dones)

        # Loss and update
        loss = F.smooth_l1_loss(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()

        self.losses.append(loss.item())
        return loss.item()

    def update_target(self):
        """Copy policy network weights to target network."""
        self.target_net.load_state_dict(self.policy_net.state_dict())


print("DQN Agent architecture:")
agent = DQNAgent(state_dim=env.state_dim, n_actions=env.n_actions)
print(agent.policy_net)

---

## 7. 🏋️ Training Loop

We train the agent over multiple episodes. Each episode:
1. Sample a random amateur photo
2. Agent applies edits sequentially
3. Receives reward based on aesthetic improvement
4. Episode ends when agent selects `done` or reaches max steps

In [ ]:
def train_agent(agent, env, n_episodes=300, verbose_every=50):
    """Train the DQN agent on photo editing task."""

    episode_rewards = []
    episode_lengths = []
    episode_scores_before = []
    episode_scores_after = []
    epsilons = []

    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        score_before = env.prev_score
        done = False

        while not done:
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)

            agent.buffer.push(state, action, reward, next_state, float(done))
            agent.update()

            state = next_state
            total_reward += reward

        # Update target network periodically
        if episode % agent.target_update == 0:
            agent.update_target()

        score_after, _ = scorer.compute_score(env.current_image)

        episode_rewards.append(total_reward)
        episode_lengths.append(env.step_count)
        episode_scores_before.append(score_before)
        episode_scores_after.append(score_after)
        epsilons.append(agent.get_epsilon())

        if (episode + 1) % verbose_every == 0:
            avg_reward = np.mean(episode_rewards[-50:])
            avg_improvement = np.mean(np.array(episode_scores_after[-50:]) -
                                     np.array(episode_scores_before[-50:]))
            print(f"Episode {episode+1}/{n_episodes} | "
                  f"Avg Reward: {avg_reward:.4f} | "
                  f"Avg Improvement: {avg_improvement:.4f} | "
                  f"Epsilon: {agent.get_epsilon():.3f} | "
                  f"Avg Length: {np.mean(episode_lengths[-50:]):.1f}")

    metrics = {
        'rewards': episode_rewards,
        'lengths': episode_lengths,
        'scores_before': episode_scores_before,
        'scores_after': episode_scores_after,
        'epsilons': epsilons,
        'losses': agent.losses
    }
    return metrics

print("Starting training...")
metrics = train_agent(agent, env, n_episodes=300, verbose_every=50)
print("\nTraining complete!")

---

## 8. 📈 Training Curves & Analysis

In [ ]:
def plot_training_curves(metrics):
    """Plot comprehensive training metrics."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Smoothing function
    def smooth(data, window=20):
        if len(data) < window:
            return data
        return np.convolve(data, np.ones(window)/window, mode='valid')

    # Episode Rewards
    ax = axes[0, 0]
    ax.plot(metrics['rewards'], alpha=0.3, color='blue', label='Raw')
    smoothed = smooth(metrics['rewards'])
    ax.plot(range(len(smoothed)), smoothed, color='blue', linewidth=2, label='Smoothed')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total Reward')
    ax.set_title('Episode Rewards', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Aesthetic Score Improvement
    ax = axes[0, 1]
    improvements = np.array(metrics['scores_after']) - np.array(metrics['scores_before'])
    ax.plot(improvements, alpha=0.3, color='green')
    smoothed_imp = smooth(improvements)
    ax.plot(range(len(smoothed_imp)), smoothed_imp, color='green', linewidth=2)
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Score Improvement')
    ax.set_title('Aesthetic Score Improvement', fontweight='bold')
    ax.grid(True, alpha=0.3)

    # Episode Lengths
    ax = axes[0, 2]
    ax.plot(metrics['lengths'], alpha=0.3, color='orange')
    smoothed_len = smooth(metrics['lengths'])
    ax.plot(range(len(smoothed_len)), smoothed_len, color='orange', linewidth=2)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Steps')
    ax.set_title('Episode Length', fontweight='bold')
    ax.grid(True, alpha=0.3)

    # Training Loss
    ax = axes[1, 0]
    if len(metrics['losses']) > 0:
        losses = metrics['losses']
        ax.plot(losses, alpha=0.2, color='red')
        smoothed_loss = smooth(losses, window=50)
        ax.plot(range(len(smoothed_loss)), smoothed_loss, color='red', linewidth=2)
    ax.set_xlabel('Update Step')
    ax.set_ylabel('Loss')
    ax.set_title('DQN Training Loss', fontweight='bold')
    ax.grid(True, alpha=0.3)

    # Epsilon Decay
    ax = axes[1, 1]
    ax.plot(metrics['epsilons'], color='purple', linewidth=2)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Epsilon')
    ax.set_title('Exploration Rate (Epsilon)', fontweight='bold')
    ax.grid(True, alpha=0.3)

    # Score Distribution (Before vs After)
    ax = axes[1, 2]
    # Last 100 episodes
    n_show = min(100, len(metrics['scores_before']))
    ax.hist(metrics['scores_before'][-n_show:], bins=20, alpha=0.5,
            label='Before Editing', color='red')
    ax.hist(metrics['scores_after'][-n_show:], bins=20, alpha=0.5,
            label='After Editing', color='green')
    ax.set_xlabel('Aesthetic Score')
    ax.set_ylabel('Count')
    ax.set_title('Score Distribution (Last 100 Episodes)', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.suptitle('Training Metrics Dashboard', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

plot_training_curves(metrics)

---

## 9. 🎨 Results & Demonstrations

### Before/After Editing Comparisons

In [ ]:
def evaluate_agent(agent, env, n_episodes=8):
    """Evaluate the trained agent and collect editing results."""
    results = []

    for _ in range(n_episodes):
        state = env.reset()
        original = env.original_image.copy()
        score_before, details_before = scorer.compute_score(original)

        done = False
        actions_taken = []
        intermediate_images = [original.copy()]
        intermediate_scores = [score_before]

        while not done:
            action = agent.select_action(state, eval_mode=True)
            state, reward, done, info = env.step(action)
            actions_taken.append(action)
            intermediate_images.append(env.current_image.copy())
            s, _ = scorer.compute_score(env.current_image)
            intermediate_scores.append(s)

        edited = env.current_image.copy()
        score_after, details_after = scorer.compute_score(edited)

        results.append({
            'original': original,
            'edited': edited,
            'score_before': score_before,
            'score_after': score_after,
            'details_before': details_before,
            'details_after': details_after,
            'actions': actions_taken,
            'intermediates': intermediate_images,
            'intermediate_scores': intermediate_scores
        })

    return results

results = evaluate_agent(agent, env, n_episodes=8)
print(f"Evaluated {len(results)} episodes.")
avg_improvement = np.mean([r['score_after'] - r['score_before'] for r in results])
print(f"Average score improvement: {avg_improvement:.4f}")

In [ ]:
# Before/After Grid
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for i in range(min(8, len(results))):
    row = (i // 4) * 2
    col = i % 4

    r = results[i]

    # Original
    axes[row, col].imshow(r['original'])
    axes[row, col].set_title(f"Before: {r['score_before']:.3f}", fontsize=10, color='red')
    axes[row, col].axis('off')

    # Edited
    axes[row+1, col].imshow(r['edited'])
    improvement = r['score_after'] - r['score_before']
    color = 'green' if improvement > 0 else 'red'
    axes[row+1, col].set_title(f"After: {r['score_after']:.3f} ({improvement:+.3f})",
                                fontsize=10, color=color)
    axes[row+1, col].axis('off')

plt.suptitle('Before/After Editing Comparisons\n(Top: Original, Bottom: Agent-Edited)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Edit Sequence Visualization

Let's visualize the step-by-step editing process for a single image:

In [ ]:
# Show detailed editing sequence for best result
best_idx = np.argmax([r['score_after'] - r['score_before'] for r in results])
best = results[best_idx]

n_steps = min(len(best['intermediates']), 12)
cols = min(n_steps, 6)
rows = (n_steps + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3.5*rows))
if rows == 1:
    axes = axes.reshape(1, -1)

for i in range(n_steps):
    row, col = i // cols, i % cols
    axes[row, col].imshow(best['intermediates'][i])

    if i == 0:
        title = f"Step 0: Original\nScore: {best['intermediate_scores'][i]:.3f}"
    else:
        action_name = env.get_action_name(best['actions'][i-1])
        title = f"Step {i}: {action_name}\nScore: {best['intermediate_scores'][i]:.3f}"

    axes[row, col].set_title(title, fontsize=8)
    axes[row, col].axis('off')

# Hide remaining axes
for i in range(n_steps, rows * cols):
    row, col = i // cols, i % cols
    axes[row, col].axis('off')

plt.suptitle('Best Edit Sequence (Step-by-Step)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Score progression
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
scores = best['intermediate_scores'][:n_steps]
ax.plot(scores, 'bo-', markersize=8, linewidth=2)
ax.fill_between(range(len(scores)), scores[0], scores, alpha=0.2, color='blue')

for i, s in enumerate(scores):
    if i > 0 and i <= len(best['actions']):
        ax.annotate(env.get_action_name(best['actions'][i-1]).replace('_', '\n'),
                   (i, s), fontsize=7, ha='center', va='bottom')

ax.set_xlabel('Editing Step', fontsize=12)
ax.set_ylabel('Aesthetic Score', fontsize=12)
ax.set_title('Aesthetic Score Progression During Best Edit Sequence', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Action Distribution Analysis

In [ ]:
# Analyze action distribution across all evaluated episodes
all_actions = []
for r in results:
    all_actions.extend(r['actions'])

action_counts = np.zeros(env.n_actions)
for a in all_actions:
    action_counts[a] += 1

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Action frequency
ax = axes[0]
colors = plt.cm.Set3(np.linspace(0, 1, env.n_actions))
bars = ax.barh(range(env.n_actions), action_counts, color=colors)
ax.set_yticks(range(env.n_actions))
ax.set_yticklabels([n.replace('_', ' ') for n in env.ACTION_NAMES], fontsize=9)
ax.set_xlabel('Frequency')
ax.set_title('Action Distribution (Evaluation)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Per-metric improvement breakdown
ax = axes[1]
metric_names = ['colorfulness', 'contrast', 'sharpness', 'exposure', 'rule_of_thirds']
improvements_per_metric = {m: [] for m in metric_names}

for r in results:
    for m in metric_names:
        improvements_per_metric[m].append(r['details_after'][m] - r['details_before'][m])

means = [np.mean(improvements_per_metric[m]) for m in metric_names]
stds = [np.std(improvements_per_metric[m]) for m in metric_names]

x_pos = range(len(metric_names))
bars = ax.bar(x_pos, means, yerr=stds, color=plt.cm.coolwarm(np.linspace(0.2, 0.8, len(metric_names))),
              capsize=5, edgecolor='black', linewidth=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([m.replace('_', '\n') for m in metric_names], fontsize=9)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_ylabel('Mean Improvement')
ax.set_title('Per-Metric Score Improvement', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Best Edits Showcase

In [ ]:
# Run more evaluations to find best edits
showcase_results = evaluate_agent(agent, env, n_episodes=20)

# Sort by improvement
improvements = [(i, r['score_after'] - r['score_before']) for i, r in enumerate(showcase_results)]
improvements.sort(key=lambda x: x[1], reverse=True)

# Show top 6 improvements
fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(3, 6, figure=fig, hspace=0.4, wspace=0.3)

for rank, (idx, imp) in enumerate(improvements[:6]):
    r = showcase_results[idx]
    col = rank

    # Original
    ax1 = fig.add_subplot(gs[0, col])
    ax1.imshow(r['original'])
    ax1.set_title(f"Original\n{r['score_before']:.3f}", fontsize=9)
    ax1.axis('off')

    # Edited
    ax2 = fig.add_subplot(gs[1, col])
    ax2.imshow(r['edited'])
    ax2.set_title(f"Edited\n{r['score_after']:.3f} ({imp:+.3f})", fontsize=9, color='green')
    ax2.axis('off')

    # Action sequence
    ax3 = fig.add_subplot(gs[2, col])
    action_names = [env.get_action_name(a)[:8] for a in r['actions'] if a != 12]
    if len(action_names) > 5:
        action_names = action_names[:5] + ['...']
    action_text = '\n'.join([f"{i+1}. {n}" for i, n in enumerate(action_names)])
    ax3.text(0.1, 0.9, action_text, transform=ax3.transAxes, fontsize=7,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax3.set_title('Actions', fontsize=9)
    ax3.axis('off')

plt.suptitle('🏆 Top 6 Best Editing Results', fontsize=16, fontweight='bold')
plt.show()

### Detailed Metric Radar Charts

In [ ]:
# Radar chart comparing before/after metrics for top results

def radar_chart(ax, categories, values_before, values_after, title):
    """Draw a radar/spider chart comparing before and after."""
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    values_before = list(values_before) + [values_before[0]]
    values_after = list(values_after) + [values_after[0]]

    ax.plot(angles, values_before, 'o-', color='red', linewidth=2, label='Before', markersize=4)
    ax.fill(angles, values_before, alpha=0.15, color='red')
    ax.plot(angles, values_after, 'o-', color='green', linewidth=2, label='After', markersize=4)
    ax.fill(angles, values_after, alpha=0.15, color='green')

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=15)
    ax.legend(loc='upper right', fontsize=7)

fig, axes = plt.subplots(1, 4, figsize=(18, 5), subplot_kw=dict(projection='polar'))

categories = ['Color', 'Contrast', 'Sharp', 'Exposure', 'RoT']
metric_keys = ['colorfulness', 'contrast', 'sharpness', 'exposure', 'rule_of_thirds']

for i in range(4):
    idx = improvements[i][0]
    r = showcase_results[idx]

    v_before = [r['details_before'][k] for k in metric_keys]
    v_after = [r['details_after'][k] for k in metric_keys]

    radar_chart(axes[i], categories, v_before, v_after, f'Example {i+1}')

plt.suptitle('Per-Metric Before/After Comparison (Radar Charts)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Q-Value Analysis

In [ ]:
# Visualize Q-values for different image states
def visualize_q_values(agent, env, n_samples=6):
    """Visualize Q-values for different image states."""
    fig, axes = plt.subplots(2, n_samples, figsize=(3*n_samples, 7))

    for i in range(n_samples):
        state = env.reset()

        axes[0, i].imshow(env.current_image)
        axes[0, i].axis('off')
        score, _ = scorer.compute_score(env.current_image)
        axes[0, i].set_title(f'Score: {score:.3f}', fontsize=9)

        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = agent.policy_net(state_tensor).cpu().numpy().flatten()

        colors = ['green' if q > 0 else 'red' for q in q_values]
        axes[1, i].barh(range(env.n_actions), q_values, color=colors, alpha=0.7)
        axes[1, i].set_yticks(range(env.n_actions))
        if i == 0:
            axes[1, i].set_yticklabels([n[:10] for n in env.ACTION_NAMES], fontsize=7)
        else:
            axes[1, i].set_yticklabels([])
        axes[1, i].axvline(x=0, color='black', linewidth=0.5)
        axes[1, i].set_xlabel('Q-value', fontsize=8)

        best_action = np.argmax(q_values)
        axes[1, i].set_title(f'Best: {env.ACTION_NAMES[best_action][:12]}', fontsize=8)

    plt.suptitle('Q-Value Landscape for Different Image States', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_q_values(agent, env)

### Editing Strategy Analysis

In [ ]:
# Analyze common editing patterns
all_sequences = [r['actions'] for r in showcase_results]

# First action distribution
first_actions = [seq[0] for seq in all_sequences if len(seq) > 0]
first_action_counts = np.zeros(env.n_actions)
for a in first_actions:
    first_action_counts[a] += 1

# Action transition matrix
transition_matrix = np.zeros((env.n_actions, env.n_actions))
for seq in all_sequences:
    for i in range(len(seq) - 1):
        transition_matrix[seq[i], seq[i+1]] += 1

# Normalize rows
row_sums = transition_matrix.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
transition_probs = transition_matrix / row_sums

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# First action preference
ax = axes[0]
sorted_idx = np.argsort(first_action_counts)[::-1]
top_k = 8
ax.bar(range(top_k), first_action_counts[sorted_idx[:top_k]],
       color=plt.cm.tab10(np.linspace(0, 1, top_k)))
ax.set_xticks(range(top_k))
ax.set_xticklabels([env.ACTION_NAMES[i][:12] for i in sorted_idx[:top_k]],
                   rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Count')
ax.set_title('First Action Preference', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Transition matrix heatmap
ax = axes[1]
im = ax.imshow(transition_probs, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.5)
ax.set_xticks(range(env.n_actions))
ax.set_yticks(range(env.n_actions))
short_names = [n[:6] for n in env.ACTION_NAMES]
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(short_names, fontsize=7)
ax.set_xlabel('Next Action')
ax.set_ylabel('Current Action')
ax.set_title('Action Transition Probabilities', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

---

## 10. 🔬 Advanced Analysis: Ablation Study

Let's study how different reward components affect the learned policy.

In [ ]:
# Quick ablation: train with different time penalties
time_penalties = [0.0, 0.002, 0.01]
ablation_results = {}

for tp in time_penalties:
    print(f"\nTraining with time_penalty = {tp}...")
    abl_env = PhotoEditingEnv(time_penalty=tp)
    abl_agent = DQNAgent(state_dim=abl_env.state_dim, n_actions=abl_env.n_actions,
                         epsilon_decay=300)
    abl_metrics = train_agent(abl_agent, abl_env, n_episodes=150, verbose_every=150)

    # Evaluate
    abl_eval = evaluate_agent(abl_agent, abl_env, n_episodes=10)
    avg_imp = np.mean([r['score_after'] - r['score_before'] for r in abl_eval])
    avg_len = np.mean([len(r['actions']) for r in abl_eval])

    ablation_results[tp] = {
        'improvement': avg_imp,
        'avg_length': avg_len,
        'rewards': abl_metrics['rewards']
    }
    print(f"  Avg improvement: {avg_imp:.4f}, Avg length: {avg_len:.1f}")

# Plot ablation results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Reward curves
ax = axes[0]
for tp, res in ablation_results.items():
    smoothed = np.convolve(res['rewards'], np.ones(15)/15, mode='valid')
    ax.plot(smoothed, label=f'\u03bb={tp}', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Smoothed Reward')
ax.set_title('Reward vs Time Penalty', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Improvement comparison
ax = axes[1]
tps = list(ablation_results.keys())
imps = [ablation_results[tp]['improvement'] for tp in tps]
ax.bar(range(len(tps)), imps, color=['#2ecc71', '#3498db', '#e74c3c'])
ax.set_xticks(range(len(tps)))
ax.set_xticklabels([f'\u03bb={tp}' for tp in tps])
ax.set_ylabel('Avg Score Improvement')
ax.set_title('Quality vs Time Penalty', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Length comparison
ax = axes[2]
lens = [ablation_results[tp]['avg_length'] for tp in tps]
ax.bar(range(len(tps)), lens, color=['#2ecc71', '#3498db', '#e74c3c'])
ax.set_xticks(range(len(tps)))
ax.set_xticklabels([f'\u03bb={tp}' for tp in tps])
ax.set_ylabel('Avg Episode Length')
ax.set_title('Efficiency vs Time Penalty', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Ablation Study: Effect of Time Penalty \u03bb', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 11. 📝 Summary

### Key Takeaways

In this project, we built a complete **autonomous photo editing system** using reinforcement learning:

1. **MDP Formulation**: Photo editing naturally maps to a sequential decision problem where the state is the current image, actions are discrete editing operations, and reward is aesthetic quality improvement.

2. **Aesthetic Quality Metrics**: We implemented computational metrics including:
   - **Colorfulness** (Hasler & Süsstrunk): $C = \sqrt{\sigma_{rg}^2 + \sigma_{yb}^2} + 0.3\sqrt{\mu_{rg}^2 + \mu_{yb}^2}$
   - **RMS Contrast**: $C_{\text{RMS}} = \sqrt{\frac{1}{N}\sum(L_i - \bar{L})^2}$
   - **Sharpness**: Laplacian variance $S = \text{Var}(\nabla^2 I)$
   - **Exposure Balance**: Penalizes deviation from optimal mean luminance
   - **Rule of Thirds**: Edge energy at composition points

3. **DQN Agent**: A Deep Q-Network with experience replay and target networks learns to select editing actions that maximize cumulative aesthetic improvement.

4. **Time Penalty**: The reward function $R_t = Q_{\text{aesthetic}}(I_t) - Q_{\text{aesthetic}}(I_{t-1}) - \lambda \cdot t$ balances quality improvement against editing efficiency.

5. **Learned Strategies**: The agent discovers meaningful editing patterns, such as adjusting exposure first, then contrast and saturation.

### Limitations & Future Work

- **Proxy reward**: Our aesthetic score is a rough approximation; real systems use neural aesthetic predictors (e.g., NIMA)
- **Continuous actions**: Real editing uses continuous parameters; future work could use continuous action RL (DDPG/SAC)
- **Higher resolution**: Scaling to full-resolution images requires CNN-based state encoding
- **User preferences**: Personalizing the reward to individual user aesthetics

### References

- Hasler, D. & Süsstrunk, S. (2003). Measuring colourfulness in natural images.
- Mnih, V. et al. (2015). Human-level control through deep reinforcement learning.
- Hu, Y. et al. (2018). Exposure: A White-Box Photo Post-Processing Framework.
- Park, J. et al. (2018). Distort-and-Recover: Color Enhancement using Deep RL.

In [ ]:
print("="*60)
print("  Module 12.2: Autonomous Photo Editor - Complete!")
print("="*60)
print("\n  Final Results:")
print("  - Training episodes: 300")
print(f"  - Average score improvement: {avg_improvement:.4f}")
print(f"  - Action space size: {env.n_actions}")
print(f"  - State dimension: {env.state_dim}")
print("\n  The agent learned to autonomously improve photo quality")
print("  by selecting appropriate editing operations in sequence.")
print("="*60)